# Non-RAG GPT-5.6 Terra prompt comparison

Run the same cancer-myth classification experiment without retrieved NCI PDQ evidence. The notebook evaluates the `basic`, `oncology_expert`, and `patient_education` prompts with the same model and strict Boolean JSON schema used by the RAG experiment.

Set `OPENAI_API_KEY` before running. Each prompt writes a separate CSV, checkpoints every completed answer, and skips question IDs already present when resumed.

In [10]:
from __future__ import annotations

from concurrent.futures import as_completed, ThreadPoolExecutor
import csv
import json
import os
from pathlib import Path
import sys
import time


def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("Could not find the project root.")


PROJECT_ROOT = find_project_root(Path.cwd())
DATASET_PATH = PROJECT_ROOT / "data" / "cancermyth_screening_dataset.json"
OUTPUT_DIR = PROJECT_ROOT / "non_rag" / "gpt-5.6-terra"

with DATASET_PATH.open(encoding="utf-8") as dataset_file:
    questions = json.load(dataset_file)

if not isinstance(questions, list) or not questions:
    raise ValueError("The dataset must be a non-empty JSON array.")
if any(not isinstance(row, dict) or "id" not in row or "question" not in row for row in questions):
    raise ValueError("Every dataset row must contain 'id' and 'question'.")
question_ids = [str(row["id"]) for row in questions]
if len(question_ids) != len(set(question_ids)):
    raise ValueError("Question IDs must be unique.")

PROMPT_KEYS = ("basic", "oncology_expert", "patient_education")
N_QUESTIONS = len(questions)  # Change to an integer to run only the first N questions.
N_WORKERS = max(1, os.cpu_count() or 4)
MAX_ATTEMPTS = 3

if isinstance(N_QUESTIONS, bool) or not isinstance(N_QUESTIONS, int):
    raise TypeError("N_QUESTIONS must be an integer.")
if not 1 <= N_QUESTIONS <= len(questions):
    raise ValueError(f"N_QUESTIONS must be between 1 and {len(questions)}.")
selected_questions = questions[:N_QUESTIONS]

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Selected questions: {len(selected_questions)} / {len(questions)}")
print(f"Available concurrent workers: {N_WORKERS}")
print(f"Output directory: {OUTPUT_DIR}")

Selected questions: 735 / 735
Available concurrent workers: 16
Output directory: D:\Personal Project\RAG In Cancer Myth\rag-cancer-myth\non_rag\gpt-5.6-terra


In [11]:
from dataclasses import replace

from rag.rag_model.config import LLMSettings
from rag.rag_model.llm import OpenAICompatibleBooleanClient
from rag.rag_model.prompts import render_prompt

settings = replace(LLMSettings.from_environment(), model="gpt-5.6-terra")
settings.validate()
if settings.model != "gpt-5.6-terra":
    raise ValueError(f"This experiment requires gpt-5.6-terra; configured model is {settings.model!r}.")
if settings.base_url.rstrip("/") == "https://api.openai.com/v1" and not settings.api_key:
    raise RuntimeError("Set OPENAI_API_KEY before running the batch.")

client = OpenAICompatibleBooleanClient(settings)
print(f"Model: {settings.model}")

Model: gpt-5.6-terra


In [12]:
CSV_COLUMNS = ["question_id", "answer"]
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def load_completed_ids(output_path: Path) -> set[str]:
    completed: set[str] = set()
    if not output_path.exists() or output_path.stat().st_size == 0:
        return completed
    with output_path.open(newline="", encoding="utf-8") as existing_file:
        reader = csv.DictReader(existing_file)
        if reader.fieldnames != CSV_COLUMNS:
            raise ValueError(f"{output_path.name} must have columns {CSV_COLUMNS}.")
        for row in reader:
            question_id = row["question_id"].strip()
            if question_id in completed:
                raise ValueError(f"Duplicate question_id in {output_path.name}: {question_id}")
            if row["answer"] not in {"true", "false"}:
                raise ValueError(f"Invalid answer for question_id={question_id} in {output_path.name}.")
            completed.add(question_id)
    return completed


def classify_with_retry(prompt_key: str, question: str, question_id: object) -> str:
    prompt = render_prompt(prompt_key, question)
    for attempt in range(1, MAX_ATTEMPTS + 1):
        try:
            value = client.classify(prompt)  # No retrieved evidence: this is the non-RAG baseline.
            return "true" if value else "false"
        except Exception:
            if attempt == MAX_ATTEMPTS:
                raise
            delay_seconds = 2 ** (attempt - 1)
            print(f"[{prompt_key}] question_id={question_id} failed; retrying in {delay_seconds}s.")
            time.sleep(delay_seconds)
    raise AssertionError("Unreachable")


def run_prompt(prompt_key: str) -> Path:
    output_path = OUTPUT_DIR / f"answers_{prompt_key}.csv"
    completed_ids = load_completed_ids(output_path)
    pending = [row for row in selected_questions if str(row["id"]) not in completed_ids]
    completed_selected = len(selected_questions) - len(pending)
    worker_count = min(N_WORKERS, max(1, len(pending)))
    print(f"\n[{prompt_key}] selected={len(selected_questions)}, completed={completed_selected}, pending={len(pending)}, workers={worker_count}")

    write_header = not output_path.exists() or output_path.stat().st_size == 0
    with output_path.open("a", newline="", encoding="utf-8") as results_file:
        writer = csv.DictWriter(results_file, fieldnames=CSV_COLUMNS)
        if write_header:
            writer.writeheader()
            results_file.flush()
            os.fsync(results_file.fileno())

        with ThreadPoolExecutor(max_workers=worker_count) as executor:
            future_to_record = {
                executor.submit(
                    classify_with_retry,
                    prompt_key,
                    str(record["question"]),
                    record["id"],
                ): record
                for record in pending
            }
            for finished, future in enumerate(as_completed(future_to_record), start=1):
                record = future_to_record[future]
                writer.writerow({"question_id": record["id"], "answer": future.result()})
                results_file.flush()
                os.fsync(results_file.fileno())
                if finished == 1 or finished % 10 == 0 or finished == len(pending):
                    print(f"[{prompt_key}] saved {finished}/{len(pending)} pending answers.")

    print(f"[{prompt_key}] complete: {output_path}")
    return output_path


output_paths = [run_prompt(prompt_key) for prompt_key in PROMPT_KEYS]


[basic] selected=735, completed=100, pending=635, workers=16
[basic] saved 1/635 pending answers.
[basic] saved 10/635 pending answers.
[basic] saved 20/635 pending answers.
[basic] saved 30/635 pending answers.
[basic] saved 40/635 pending answers.
[basic] saved 50/635 pending answers.
[basic] saved 60/635 pending answers.
[basic] saved 70/635 pending answers.
[basic] saved 80/635 pending answers.
[basic] saved 90/635 pending answers.
[basic] saved 100/635 pending answers.
[basic] saved 110/635 pending answers.
[basic] saved 120/635 pending answers.
[basic] saved 130/635 pending answers.
[basic] saved 140/635 pending answers.
[basic] saved 150/635 pending answers.
[basic] saved 160/635 pending answers.
[basic] saved 170/635 pending answers.
[basic] saved 180/635 pending answers.
[basic] saved 190/635 pending answers.
[basic] saved 200/635 pending answers.
[basic] saved 210/635 pending answers.
[basic] saved 220/635 pending answers.
[basic] saved 230/635 pending answers.
[basic] saved

In [13]:
for output_path in output_paths:
    completed_ids = load_completed_ids(output_path)
    selected_ids = {str(row["id"]) for row in selected_questions}
    missing_ids = selected_ids - completed_ids
    print(f"{output_path.name}: {len(selected_ids - missing_ids)}/{len(selected_ids)} selected questions saved")
    assert not missing_ids, f"{output_path.name} is missing selected IDs: {sorted(missing_ids)}"

answers_basic.csv: 735/735 selected questions saved
answers_oncology_expert.csv: 735/735 selected questions saved
answers_patient_education.csv: 735/735 selected questions saved
